# 04 · Semantic rule generalization

**Question:** Does a frozen semantic encoder transfer better to a new rule?

This notebook reviews completed Qwen3 embedding experiments. Restore the latest S3 snapshot first. To compute a new experiment, use `uv run --extra semantic jigsaw semantic --cloud` on a CPU workspace with at least 8 GB RAM and 4 GB currently free. Completed embedding shards are reused.

The original lexical baseline remains unchanged. Software verification uses an explicitly labeled test encoder on synthetic data; those numbers are never model-performance evidence.

In [ ]:
import os
from pathlib import Path
import json
import pandas as pd
import plotly.express as px
from IPython.display import display, HTML, FileLink
from jigsaw_rules.data import load_data, audit
from jigsaw_rules.runtime import Progress, environment

root = Path(os.environ.get("JIGSAW_ROOT", Path.cwd())).resolve()
if root.name == "notebooks":
    root = root.parent
px.defaults.template = "plotly_white"
px.defaults.color_discrete_sequence = ["#087f8c", "#bd633b", "#334ea0", "#70923b"]
display(HTML("<div style='padding:18px;background:#edf6f5;border-left:5px solid #087f8c'>"
             "<b>Jigsaw research workspace</b><br>Every result must identify its data and validation protocol.</div>"))
print("Project:", root)


In [ ]:
from jigsaw_rules.review import review_run
is_demo = (root / 'data/raw/SYNTHETIC.txt').exists()
if is_demo:
    from tests.helpers import TestEncoder
    from jigsaw_rules.embeddings import load_spec
    from jigsaw_rules.pipeline import run_baseline
    from jigsaw_rules.semantic_pipeline import run_semantic
    source = Path.cwd()
    if source.name == 'notebooks': source = source.parent
    spec = load_spec(source)
    spec['baseline_run'] = run_baseline(root).name
    run_dir = run_semantic(root, spec, encoder=TestEncoder())
    display(HTML('<b>SYNTHETIC SOFTWARE TEST — test vectors, not Qwen performance</b>'))
else:
    candidates = []
    for path in (root / 'runs').glob('*/status.json'):
        status = json.loads(path.read_text())
        if status.get('experiment') == 'semantic' and status.get('status') == 'completed' and status.get('synthetic') is False:
            finished = json.loads((path.parent / 'review/complete.json').read_text())['finished_at']
            candidates.append((finished, path.parent))
    if not candidates: raise FileNotFoundError('Restore the completed semantic experiment with uv run jigsaw restore first.')
    run_dir = max(candidates)[1]
evidence = review_run(root, run_dir.name, allow_synthetic=is_demo)
print('Data kind:', evidence['data_kind'], '| Run:', evidence['run_id'])

## Compare the same validation assignments
The semantic experiment reuses the original split files and checks their hashes, row coverage, and training-text isolation. The encoder is frozen. The scaler and classifier see only training-fold rows. The example-margin model uses a predeclared temperature of 0.1; its outputs are not claimed to be calibrated.

In [ ]:
comparison = json.loads((run_dir / 'review/comparison.json').read_text())
records = comparison['baseline'] + comparison['semantic']
summary = pd.DataFrame([{'Model': r['model'], 'Protocol': r['protocol'], **{k: v for k,v in r['metrics'].items() if isinstance(v, float)}} for r in records])
display(summary.round(4))
fig = px.bar(summary, x='Protocol', y='rule_macro_auc', color='Model', barmode='group', title='Lexical and semantic generalization on the same splits')
fig.update_yaxes(range=[0, 1])
fig.add_hline(y=.5, line_dash='dash')
fig.show()

## Uncertainty and probability quality
Paired bootstrap intervals resample normalized comment groups shared across rules. They are conditional on the two observed rules and fixed OOF predictions. They do not estimate performance across arbitrary future policies or remove model-selection bias. Inspect Brier, log loss, and calibration alongside ranking AUC.

In [ ]:
intervals = pd.DataFrame(json.loads((run_dir / 'review/uncertainty.json').read_text()))
display(intervals[['model','protocol','observed_delta','ci_lower','ci_upper','draws']].round(4))
stats = json.loads((run_dir / 'embeddings/statistics.json').read_text())
display(pd.DataFrame(stats.items(), columns=['Measurement','Value']))
print('Encoding time includes tokenization/inference; total invocation time is in performance/timing.json.')
print('Truncation counts refer to unique encoded inputs, not expanded training rows.')
display(FileLink(str(root / 'reports/private/report.html')))
display(FileLink(str(root / 'reports/private/results.json')))

## Next experiment
Use the evidence to decide whether a rule-conditioned cross-encoder or instruction model adds value. Keep the observed two-rule validation limitation visible. The current preview submission checks row alignment and probability format; a scored Kaggle submission requires the later offline inference package and account eligibility.